In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(r"C:\Users\yzhoo\AI-Agent-Job-Search")

JOB_ROOT = PROJECT_ROOT / "experiments" / "dataset" / "job_dataset"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "dataset" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert JOB_ROOT.exists(), f"Dataset folder not found: {JOB_ROOT}"

print("Job folder:", JOB_ROOT)
print("Output folder:", OUTPUT_DIR)

Job folder: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\job_dataset
Output folder: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed


In [2]:
csv_files = sorted(JOB_ROOT.rglob("*.csv"))
md_files = sorted(JOB_ROOT.rglob("*.md"))

print("Total CSV files:", len(csv_files))
print("Total documentation files:", len(md_files))

if csv_files:
    for f in csv_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  - CSV file: {f.name} ({size_mb:.2f} MB)")
else:
    print("No CSV file found.")

Total CSV files: 1
Total documentation files: 1
  - CSV file: jobstreet_all_job_dataset.csv (117.69 MB)


In [3]:
def read_csv_fallback(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode CSV: {path}")

if csv_files:
    jobs_df = read_csv_fallback(csv_files[0])
    print("CSV shape:", jobs_df.shape)
    print("Columns:", jobs_df.columns.tolist())
    display(jobs_df.head())
else:
    jobs_df = None

CSV shape: (59306, 11)
Columns: ['job_id', 'job_title', 'company', 'descriptions', 'location', 'category', 'subcategory', 'role', 'type', 'salary', 'listingDate']


,job_id,job_title,company,descriptions,location,category,subcategory,role,type,salary,listingDate
0,74630583,Procurement Executive (Contract),Coca-Cola Bottlers (Malaysia) Sdn Bhd,"Position Purpose\nManage aspects of procurement including supplier management, contract management, and vendor performance tracking. Support the administration and management of procurement processes. This includes setting up, maintaining, and administering purchasing databases; developing and issuing purchase orders; maintain contract log and providing support to other departments within the organization who are involved in procurement.\nKey Duties / Responsibilities\nGeneral\nEnsure the PO created following the bidding requirements\nOA updates from time to time - upon request from planning team / buyers\nGeographical coverage: Malaysia and Singapore operations\nEnsure all purchasing activities comply with Procurement Procedure and company internal policy\nActively seek out opportunities for improvement, ways to do things better; and lead implementation of actions defined\nManage reports and reviews\nManage reports and reviews weekly, monthly and quarterly accordingly\nContract Log Update; Ensure expiring / expired contracts are renewed on time; Safekeeping of contracts softcopy\nActive and Inactive Vendor Code Cleansing on quarterly basis\nSupplier management:\nDefine and implement strategy for supplier management to ensure reliability in consistently meeting requirements, quality control and product performance per contract, continuous improvement in cost, quality and service.\nVendor Creation follows the policy and acknowledgment from respective buyers\nDocumentation and Audit:\nKeep record of PAF, PAF approvals, Bidding documents, Contracts softcopies\nWork closely with Legal in contract storage and contract log\nKeep track on all report as evidence for audit.\nRequirements / Qualifications\nEducation:\nBachelor Degree preferably in procurement and/or supply chain related field. Professional knowledge of procurement practices, supply markets, terms, negotiation concepts and business laws.\nExperience:\n1-2 years’ experience relevant experience in FMCG/beverage manufacturing environment or any other related field with a strong procurement background in Malaysia.\nLanguage\n: Bahasa Melayu/ English\nWork Location\n: Bandar Enstek (Negeri Sembilan)\nEmployment\n: 12 months Contract\nWith Regret , only shortlisted candidate will be notified.",Negeri Sembilan,"Manufacturing, Transport & Logistics","Purchasing, Procurement & Inventory",procurement-executive,Contract/Temp,NaN,2024-03-21T05:58:35Z
1,74660602,Account Executive/ Assistant,Acoustic & Lighting System Sdn Bhd,"We are looking for a Account Executive/ Assistant to join our team.\nWhat you will get when you join us:\nFair work life balance – Mon to Friday, 9am - 6pm\nInsurance & medical coverage\nSport facilities\nResponsibilities:\nResponsible for daily accounts recording functions.\nAssist in overall filing of accounts related documents and maintenance of records.\nTo ensure the accounts payable/receivable, bank reconciliation, journal and general ledger functions are performed in an effective and efficient manner.\nResponsible for the timely submission, accuracy and maintenance of financial records.\nMonitor payment and installments.\nIssuing DN/CN, payment vouchers and journal voucher.\nThe scope of work should be revised from time to time based on management needs or requirements.\nJob Requirements:\nAt least Diploma/ Degree in Finance/Accountancy/Banking or equivalent.\nFresh Graduate are encouraged to apply.\nPrefer candidates that can speak fluent in\nMandarin\nis an added advantage.",Petaling,Accounting,Bookkeeping & Small Practice Accounting,executive-assistant,Full time,"RM 2,800 – RM 3,200 per month",2024-03-22T06:52:57Z
2,74655679,"Data Analyst - Asset Management, SPX Express",Shopee Mobile Malaysia Sdn Bhd,"Performs detailed data analysis on existing spendings / consumptions of S

In [4]:
# Verification against official Hugging Face azrai99/job-dataset specification
hf_expected_cols = [
    "job_id", "job_title", "company", "descriptions", "location",
    "category", "subcategory", "role", "type", "salary", "listingDate"
]

matches_hf = (jobs_df.columns.tolist() == hf_expected_cols)
print(f"Matches official Hugging Face azrai99/job-dataset schema exactly: {matches_hf}")

missing_summary = pd.DataFrame({
    "column": jobs_df.columns,
    "missing_count": jobs_df.isnull().sum().values,
    "missing_percent": (jobs_df.isnull().sum().values / len(jobs_df) * 100).round(2),
    "data_type": [str(t) for t in jobs_df.dtypes.values]
})

print("\nMissing values overview:")
display(missing_summary)

Matches official Hugging Face azrai99/job-dataset schema exactly: True

Missing values overview:


,column,missing_count,missing_percent,data_type
0,job_id,0,0.00,int64
1,job_title,0,0.00,object
2,company,0,0.00,object
3,descriptions,0,0.00,object
4,location,0,0.00,object
5,category,0,0.00,object
6,subcategory,0,0.00,object
7,role,1936,3.26,object
8,type,0,0.00,object
9,salary,32665,55.08,object


In [5]:
dup_ids = jobs_df["job_id"].duplicated().sum()
dup_postings = jobs_df.duplicated(subset=["job_title", "company", "location"]).sum()

desc_word_counts = jobs_df["descriptions"].astype(str).str.split().str.len()
short_desc = jobs_df[desc_word_counts < 20]

print(f"Duplicate job_id: {dup_ids}")
print(f"Duplicate postings (same title, company, location): {dup_postings} ({(dup_postings/len(jobs_df)*100):.2f}%)")
print(f"Postings needing review (< 20 words in description): {len(short_desc)}")

quality_review_df = jobs_df[
    (desc_word_counts < 20) |
    jobs_df.duplicated(subset=["job_title", "company", "location"], keep=False)
][["job_id", "job_title", "company", "location", "salary", "type"]].copy()

print(f"Total postings flagged for review: {len(quality_review_df):,d}")
display(quality_review_df.head(10))

Duplicate job_id: 0
Duplicate postings (same title, company, location): 4755 (8.02%)
Postings needing review (< 20 words in description): 159
Total postings flagged for review: 8,887


,job_id,job_title,company,location,salary,type
1,74660602,Account Executive/ Assistant,Acoustic & Lighting System Sdn Bhd,Petaling,"RM 2,800 – RM 3,200 per month",Full time
6,74677110,Accounts Executive,GENETEC TECHNOLOGY BERHAD,Kajang/Bangi/Serdang,"RM 3,000 – RM 4,500 per month",Full time
11,74662105,Finance Executive,CapitaLand Malaysia REIT Management Sdn. Bhd.,Kuala Lumpur City Centre,NaN,Full time
14,74661211,Accounts Executive,Binastra Land Sdn Bhd,Bukit Jalil,"RM 3,500 – RM 4,000 per month",Full time
16,74660694,Audit Assistant,William C.H. Tan & Associates,Petaling,NaN,Full time
19,74673679,Operations Engineer,Direct Access Solutions Sdn. Bhd.,Kuala Lumpur,"RM 5,000 – RM 7,000 per month",Full time
22,74658688,Account cum Admin Executive,Dessert Captain Sdn Bhd,Shah Alam/Subang,"RM 2,800 – RM 3,500 per month",Full time
27,74673818,Mechanical Design Engineer,GENETEC TECHNOLOGY BERHAD,Hulu Langat,"RM 3,000 – RM 4,000 per month",Full time
35,74661614,Internal Audit Executive / Senior Executive,TMK CHEMICAL BHD.,Petaling,NaN,Full time
36,74668376,Associate Data Engineer (Hybrid Working),SEEK,Kuala Lumpur,NaN,Full time


In [6]:
print("Top 10 Job Titles:")
display(
    jobs_df["job_title"].value_counts().head(10).reset_index(name="count")
)

print("\nEmployment Type (Job Type) Distribution:")
type_summary = jobs_df["type"].value_counts().reset_index(name="count")
type_summary["percent"] = (type_summary["count"] / len(jobs_df) * 100).round(2)
display(type_summary)

print("\nTop 10 Locations:")
display(
    jobs_df["location"].value_counts().head(10).reset_index(name="count")
)

Top 10 Job Titles:


,job_title,count
0,Account Executive,596
1,Accounts Executive,467
2,Sales Executive,362
3,Admin Assistant,316
4,Account Assistant,259
5,Admin Executive,228
6,Project Engineer,222
7,Marketing Executive,221
8,Finance Executive,209
9,ACCOUNT EXECUTIVE,191



Employment Type (Job Type) Distribution:


,type,count,percent
0,Full time,53790,90.70
1,Contract/Temp,4976,8.39
2,Part time,289,0.49
3,Casual/Vacation,251,0.42



Top 10 Locations:


,location,count
0,Kuala Lumpur,13712
1,Petaling,9746
2,Johor Bahru District,4480
3,Shah Alam/Subang,2574
4,Selangor,2548
5,Kuala Lumpur City Centre,2112
6,Penang Island,1885
7,Seberang Perai,1796
8,Penang,1669
9,Klang District,1427


In [7]:
print(f"Total postings with disclosed salary: {jobs_df['salary'].notna().sum():,d} ({(jobs_df['salary'].notna().mean()*100):.2f}%)")
print(f"Total postings with undisclosed salary: {jobs_df['salary'].isna().sum():,d} ({(jobs_df['salary'].isna().mean()*100):.2f}%)")

# Sample raw salary values exhibiting encoding artifacts
print("\nSample raw salary values:")
for val in jobs_df["salary"].dropna().head(5):
    print(" ", repr(val))

def clean_salary_string(s):
    if pd.isna(s):
        return None, None
    clean = str(s).replace("\xa0", " ").replace("\ufffd", "-").replace("", "-")
    m = re.search(r"RM\s*([\d,]+)\s*-\s*RM\s*([\d,]+)", clean, re.IGNORECASE)
    if m:
        min_sal = float(m.group(1).replace(",", ""))
        max_sal = float(m.group(2).replace(",", ""))
        return min_sal, max_sal
    return None, None

sample_cleaned = jobs_df["salary"].dropna().head(10).apply(clean_salary_string)
cleaned_df = pd.DataFrame(sample_cleaned.tolist(), columns=["parsed_min_salary", "parsed_max_salary"])
cleaned_df["raw_salary"] = jobs_df["salary"].dropna().head(10).values

print("\nParsed salary sample (normalized numeric ranges):")
display(cleaned_df)

Total postings with disclosed salary: 26,641 (44.92%)
Total postings with undisclosed salary: 32,665 (55.08%)

Sample raw salary values:
  'RM\xa02,800 – RM\xa03,200 per month'
  'RM\xa03,000 – RM\xa03,500 per month'
  'RM\xa02,800 – RM\xa03,500 per month'
  'RM\xa03,000 – RM\xa04,500 per month'
  'RM\xa02,000 – RM\xa03,000 per month'

Parsed salary sample (normalized numeric ranges):


,parsed_min_salary,parsed_max_salary,raw_salary
0,None,None,"RM 2,800 – RM 3,200 per month"
1,None,None,"RM 3,000 – RM 3,500 per month"
2,None,None,"RM 2,800 – RM 3,500 per month"
3,None,None,"RM 3,000 – RM 4,500 per month"
4,None,None,"RM 2,000 – RM 3,000 per month"
5,None,None,"RM 3,000 – RM 3,200 per month"
6,None,None,"RM 1,600 – RM 2,000 per month"
7,None,None,"RM 3,500 – RM 4,000 per month"
8,None,None,"RM 3,000 – RM 4,000 per month"
9,None,None,"RM 3,500 – RM 5,000 per month"


In [8]:
print("Job Description Word Count Statistics:")
print(desc_word_counts.describe().to_string())

sample_job = jobs_df.iloc[0]
print(f"\nSample Job ID: {sample_job['job_id']}")
print(f"Title: {sample_job['job_title']} at {sample_job['company']}")
print(f"Location: {sample_job['location']}")
print(f"Salary: {sample_job['salary']}")
print("\n--- Description Preview ---\n")
print(str(sample_job["descriptions"])[:1500])

Job Description Word Count Statistics:
count    59306.000000
mean       262.114238
std        169.898497
min          1.000000
25%        152.000000
50%        223.000000
75%        327.000000
max       2063.000000

Sample Job ID: 74630583
Title: Procurement Executive (Contract) at Coca-Cola Bottlers (Malaysia) Sdn Bhd
Location: Negeri Sembilan
Salary: nan

--- Description Preview ---

Position Purpose
Manage aspects of procurement including supplier management, contract management, and vendor performance tracking. Support the administration and management of procurement processes. This includes setting up, maintaining, and administering purchasing databases; developing and issuing purchase orders; maintain contract log and providing support to other departments within the organization who are involved in procurement.
Key Duties / Responsibilities
General
Ensure the PO created following the bidding requirements
OA updates from time to time - upon request from planning team / buyers
Geo

In [9]:
# Export category summary
category_summary = jobs_df.groupby("category").size().reset_index(name="job_count").sort_values("job_count", ascending=False)
category_summary.to_csv(OUTPUT_DIR / "job_category_summary.csv", index=False, encoding="utf-8-sig")

# Export location summary
location_summary = jobs_df.groupby("location").size().reset_index(name="job_count").sort_values("job_count", ascending=False)
location_summary.to_csv(OUTPUT_DIR / "job_location_summary.csv", index=False, encoding="utf-8-sig")

# Export quality review records
quality_review_df.to_csv(OUTPUT_DIR / "job_quality_review.csv", index=False, encoding="utf-8-sig")

print("Saved processed files to:", OUTPUT_DIR)

Saved processed files to: C:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed


In [10]:
print("===== JOB DATASET SUMMARY =====")
print("Total job records:", len(jobs_df))
print("Unique job titles:", jobs_df["job_title"].nunique())
print("Unique categories:", jobs_df["category"].nunique())
print("Unique locations:", jobs_df["location"].nunique())
print("Salary disclosed rate:", f"{(jobs_df['salary'].notna().mean()*100):.2f}%")
print("Full-time postings:", f"{(jobs_df['type'] == 'Full time').mean()*100:.2f}%")
print("Postings needing review:", len(quality_review_df))

display(category_summary.head(15))

===== JOB DATASET SUMMARY =====
Total job records: 59306
Unique job titles: 34686
Unique categories: 30
Unique locations: 204
Salary disclosed rate: 44.92%
Full-time postings: 90.70%
Postings needing review: 8887


,category,job_count
0,Accounting,9983
11,Engineering,8092
17,Information & Communication Technology,7872
1,Administration & Office Support,6420
25,Sales,4720
20,"Manufacturing, Transport & Logistics",4253
16,Human Resources & Recruitment,3714
21,Marketing & Communications,2451
3,Banking & Financial Services,2362
5,Call Centre & Customer Service,1465
